# STEP Sentinel-2 2016–2017 QA60 Rebuild and Earth Engine Re-ingestion

Rebuild the historical Sentinel-2 Earth Engine ImageCollection from the validated 12-band local TIFFs containing reconstructed QA60.

Sequence:
1. Validate 562 local 12-band TIFFs.
2. Overwrite the existing GCS objects at the same URIs.
3. Verify GCS byte sizes.
4. Inventory and delete existing Earth Engine child images while retaining the collection container.
5. Regenerate 12-band manifests with SAMPLE pyramiding for SCL and QA60.
6. Test-ingest one image and verify the cloud-probability join.
7. Re-ingest the full collection.
8. Run final collection and cloud-probability QA.


In [1]:
from pathlib import Path
import json
import re
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm

import ee
from google.cloud import storage
import google.auth
from google.auth.exceptions import DefaultCredentialsError

PROJECT_ID = "bop-nca-data-space"

EE_COLLECTION_ID = (
    "projects/bop-nca-data-space/"
    "assets/S2_C1_L2A_2016_2017"
)

GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"
GCS_LOCATION = "us-central1"
GCS_PREFIX = "s2-c1-2016-2017/geotiffs"

GEE_READY_DIR = Path(
    r"A:\NCA_DATA\S2_2016-2017\gee_ready"
)

MANIFEST_DIR = Path(
    r"A:\NCA_DATA\S2_2016-2017\ee_manifests_qa60"
)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_BANDS = [
    "B2","B3","B4","B5","B6","B7",
    "B8","B8A","B11","B12","SCL","QA60"
]

EXPECTED_COUNT = 562
UPLOAD_CHUNK_SIZE = 64 * 1024 * 1024

print("Project:", PROJECT_ID)
print("EE collection:", EE_COLLECTION_ID)
print("Local source:", GEE_READY_DIR)
print("Expected bands:", EXPECTED_BANDS)


Project: bop-nca-data-space
EE collection: projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017
Local source: A:\NCA_DATA\S2_2016-2017\gee_ready
Expected bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL', 'QA60']


## 1. Authenticate Google Cloud and Earth Engine

In [2]:
try:
    credentials, detected_project = google.auth.default()
    print("ADC available.")
    print("Detected project:", detected_project)
except DefaultCredentialsError:
    raise RuntimeError(
        "Application Default Credentials not found. "
        "Run gcloud auth application-default login first."
    )

storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=credentials
)

bucket = storage_client.bucket(GCS_BUCKET)

ee.Initialize(project=PROJECT_ID)

print("Google Cloud and Earth Engine initialized.")


ADC available.
Detected project: None
Google Cloud and Earth Engine initialized.


## 2. Validate all local corrected TIFFs

Hard stop: all 562 products must validate before overwrite.


In [3]:
def mget(meta, *keys, required=True, default=None):
    for key in keys:
        if key in meta and meta[key] not in (None, ""):
            return meta[key]
    if required:
        raise KeyError(f"Missing required metadata field; tried {keys}")
    return default


records = []
errors = []

for tif_path in tqdm(
    sorted(GEE_READY_DIR.glob("*_GEE.tif")),
    desc="Validating 12-band TIFFs",
):
    json_path = tif_path.with_suffix(".json")

    if not json_path.exists():
        errors.append({
            "file": tif_path.name,
            "error": "Missing JSON sidecar",
        })
        continue

    try:
        meta = json.loads(
            json_path.read_text(encoding="utf-8")
        )

        with rasterio.open(tif_path) as src:
            if src.count != 12:
                raise ValueError(
                    f"Expected 12 bands; found {src.count}"
                )

            if list(src.descriptions) != EXPECTED_BANDS:
                raise ValueError(
                    f"Unexpected band order: {src.descriptions}"
                )

            if any(dtype != "uint16" for dtype in src.dtypes):
                raise ValueError(
                    f"Unexpected dtypes: {src.dtypes}"
                )

            qa_values = set(
                np.unique(src.read(12)).tolist()
            )

            if not qa_values <= {0,1024,2048,3072}:
                raise ValueError(
                    f"Unexpected QA60 values: {qa_values}"
                )

        records.append({
            "product_id": mget(meta, "product_id", "PRODUCT_ID"),
            "sensing_time": mget(meta, "sensing_time", "SENSING_TIME"),
            "mgrs_tile": mget(meta, "mgrs_tile", "MGRS_TILE"),
            "processing_baseline": mget(
                meta, "processing_baseline", "PROCESSING_BASELINE"
            ),
            "satellite": mget(meta, "satellite", "SATELLITE"),
            "source": mget(
                meta, "source", "SOURCE",
                required=False,
                default="CDSE_COLLECTION1_L2A",
            ),
            "tif_path": tif_path,
            "json_path": json_path,
            "file_name": tif_path.name,
            "file_size_bytes": tif_path.stat().st_size,
        })

    except Exception as exc:
        errors.append({
            "file": tif_path.name,
            "error": repr(exc),
        })


inventory = pd.DataFrame(records)

print("Valid corrected products:", len(inventory))
print("Expected:", EXPECTED_COUNT)
print("Validation errors:", len(errors))

if len(inventory):
    print(
        "Prepared volume:",
        f"{inventory.file_size_bytes.sum()/1024**3:,.2f} GiB"
    )

if errors:
    display(pd.DataFrame(errors))

if len(inventory) != EXPECTED_COUNT or errors:
    raise RuntimeError(
        "Local validation failed. Do not proceed to GCS overwrite."
    )

print("Local validation PASSED.")


Validating 12-band TIFFs:   0%|          | 0/562 [00:00<?, ?it/s]

Valid corrected products: 562
Expected: 562
Validation errors: 0
Prepared volume: 182.02 GiB
Local validation PASSED.


## 3. Overwrite existing GCS objects

Set `OVERWRITE_GCS = True` after local validation passes. Object names stay unchanged.


In [4]:
def object_name(file_name):
    return f"{GCS_PREFIX}/{file_name}"

OVERWRITE_GCS = True
print("GCS overwrite enabled:", OVERWRITE_GCS)


GCS overwrite enabled: True


In [5]:
overwrite_results = []

if not OVERWRITE_GCS:
    print("GCS overwrite disabled.")
else:
    for _, row in tqdm(
        inventory.iterrows(),
        total=len(inventory),
        desc="Overwriting GCS objects",
    ):
        local = Path(row.tif_path)
        remote_name = object_name(local.name)
        local_size = int(row.file_size_bytes)

        blob = bucket.blob(remote_name)
        blob.chunk_size = UPLOAD_CHUNK_SIZE

        try:
            blob.upload_from_filename(
                str(local),
                timeout=900,
            )
            blob.reload()

            remote_size = int(blob.size or 0)

            if remote_size != local_size:
                raise RuntimeError(
                    "Remote byte size does not match local TIFF: "
                    f"local={local_size}, remote={remote_size}"
                )

            overwrite_results.append({
                "file": local.name,
                "status": "overwritten",
                "local_bytes": local_size,
                "remote_bytes": remote_size,
                "generation": blob.generation,
            })

        except Exception as exc:
            overwrite_results.append({
                "file": local.name,
                "status": "failed",
                "error": repr(exc),
            })

if overwrite_results:
    overwrite_df = pd.DataFrame(overwrite_results)
    display(
        overwrite_df.status.value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    failed = overwrite_df.loc[
        overwrite_df.status.eq("failed")
    ]

    print("Failed GCS overwrites:", len(failed))

    if len(failed):
        display(failed)


Overwriting GCS objects:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,overwritten,562


Failed GCS overwrites: 0


## 4. Verify GCS staging

In [6]:
remote_objects = {
    blob.name: {
        "size": int(blob.size or 0),
        "generation": blob.generation,
    }
    for blob in storage_client.list_blobs(
        GCS_BUCKET,
        prefix=GCS_PREFIX + "/",
    )
}

verification_rows = []

for _, row in inventory.iterrows():
    remote_name = object_name(row.file_name)
    remote = remote_objects.get(remote_name)
    local_size = int(row.file_size_bytes)

    verification_rows.append({
        "file": row.file_name,
        "present": remote is not None,
        "local_bytes": local_size,
        "remote_bytes": (
            None if remote is None else remote["size"]
        ),
        "size_match": (
            remote is not None
            and remote["size"] == local_size
        ),
    })

gcs_verify = pd.DataFrame(verification_rows)

print("Expected objects:", len(inventory))
print("Present:", int(gcs_verify.present.sum()))
print("Exact size matches:", int(gcs_verify.size_match.sum()))

problems = gcs_verify.loc[~gcs_verify.size_match]
print("Problems:", len(problems))

if len(problems):
    display(problems)


Expected objects: 562
Present: 562
Exact size matches: 562
Problems: 0


## 5. Inventory existing Earth Engine child images

The ImageCollection container is retained.


In [7]:
def list_all_child_assets(parent):
    assets = []
    page_token = None

    while True:
        params = {
            "parent": parent,
            "pageSize": 1000,
        }

        if page_token:
            params["pageToken"] = page_token

        response = ee.data.listAssets(params)

        assets.extend(
            response.get("assets", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return assets


existing_children = list_all_child_assets(
    EE_COLLECTION_ID
)

print(
    "Existing EE child assets:",
    len(existing_children)
)

if existing_children:
    display(
        pd.DataFrame([
            {
                "id": a.get("id"),
                "type": a.get("type"),
            }
            for a in existing_children
        ]).head(20)
    )

Existing EE child assets: 562


,id,type
0,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
1,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
2,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
3,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
4,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
5,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
6,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
7,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
8,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE
9,projects/bop-nca-data-space/assets/S2_C1_L2A_2...,IMAGE


## 6. Delete existing Earth Engine child images

Review the inventory first. Then set `CONFIRM_DELETE_OLD_IMAGES = True`.


In [8]:
CONFIRM_DELETE_OLD_IMAGES = True

delete_results = []

if not CONFIRM_DELETE_OLD_IMAGES:
    print("Deletion disabled.")
    print(
        "Assets that WOULD be deleted:",
        len(existing_children)
    )
else:
    for asset in tqdm(
        existing_children,
        desc="Deleting old EE images",
    ):
        asset_id = asset["id"]

        try:
            ee.data.deleteAsset(asset_id)

            delete_results.append({
                "asset_id": asset_id,
                "status": "deleted",
            })

        except Exception as exc:
            delete_results.append({
                "asset_id": asset_id,
                "status": "failed",
                "error": repr(exc),
            })

    delete_df = pd.DataFrame(delete_results)

    display(
        delete_df.status.value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    failed_delete = delete_df.loc[
        delete_df.status.eq("failed")
    ]

    print("Failed deletions:", len(failed_delete))

    if len(failed_delete):
        display(failed_delete)


Deleting old EE images:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,deleted,562


Failed deletions: 0


## 7. Verify the collection is empty before re-ingestion

In [9]:
remaining_children = list_all_child_assets(
    EE_COLLECTION_ID
)

print(
    "Remaining child assets:",
    len(remaining_children)
)


Remaining child assets: 0


## 8. Build corrected 12-band manifests

In [10]:
def rfc3339(value):
    dt = datetime.fromisoformat(
        str(value).replace("Z", "+00:00")
    )

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return (
        dt.astimezone(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )


def safe_asset_name(value):
    return re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        str(value),
    )


def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False


def make_manifest(row):
    pid = str(row.product_id)
    timestamp = rfc3339(row.sensing_time)

    asset_id = (
        f"{EE_COLLECTION_ID}/"
        f"{safe_asset_name(pid)}"
    )

    uri = (
        f"gs://{GCS_BUCKET}/"
        f"{object_name(row.file_name)}"
    )

    bands = []

    for i, band in enumerate(EXPECTED_BANDS):
        bands.append({
            "id": band,
            "tilesetId": "0",
            "tilesetBandIndex": i,
            "pyramidingPolicy": (
                "SAMPLE"
                if band in {"SCL", "QA60"}
                else "MEAN"
            ),
        })

    return {
        "name": asset_id,

        "tilesets": [{
            "id": "0",
            "sources": [{
                "uris": [uri]
            }],
        }],

        "bands": bands,

        "startTime": timestamp,
        "endTime": timestamp,

        "properties": {
            "PRODUCT_ID": pid,
            "MGRS_TILE": str(row.mgrs_tile),
            "PROCESSING_BASELINE": str(
                row.processing_baseline
            ),
            "SATELLITE": str(row.satellite),
            "source": str(row.source),
            "source_group": "historical_cdse",
            "QA60_RECONSTRUCTED": "true",
        },
    }


manifest_rows = []

for _, row in tqdm(
    inventory.iterrows(),
    total=len(inventory),
    desc="Writing corrected manifests",
):
    manifest = make_manifest(row)

    manifest_path = (
        MANIFEST_DIR
        / (
            safe_asset_name(row.product_id)
            + ".json"
        )
    )

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
        ),
        encoding="utf-8",
    )

    manifest_rows.append({
        "product_id": row.product_id,
        "asset_id": manifest["name"],
        "manifest_path": manifest_path,
    })


manifest_df = pd.DataFrame(manifest_rows)

print(
    "Corrected manifests:",
    len(manifest_df)
)

if len(manifest_df) != EXPECTED_COUNT:
    raise RuntimeError(
        "Manifest count does not match expected product count."
    )


Writing corrected manifests:   0%|          | 0/562 [00:00<?, ?it/s]

Corrected manifests: 562


## 9. Test-ingest one corrected image

In [45]:
TEST_INDEX = 0

test_manifest_path = Path(
    manifest_df.iloc[
        TEST_INDEX
    ].manifest_path
)

test_manifest = json.loads(
    test_manifest_path.read_text(
        encoding="utf-8"
    )
)

test_asset_id = test_manifest["name"]

print("Test asset:", test_asset_id)

if asset_exists(test_asset_id):
    print(
        "Test asset already exists; "
        "no new task submitted."
    )
    test_operation_name = None
else:
    response = ee.data.startIngestion(
        None,
        test_manifest,
    )

    print("Response:", response)

    test_operation_name = response.get(
        "name"
    )

    print(
        "Operation:",
        test_operation_name
    )


Test asset: projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017/S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631
Test asset already exists; no new task submitted.


## 10. Inspect the test ingestion operation

In [44]:
if test_operation_name is None:
    print("No test operation to inspect.")
else:
    op = ee.data.getOperation(
        test_operation_name
    )

    print(
        json.dumps(
            op,
            indent=2,
            default=str,
        )
    )


{
  "name": "projects/bop-nca-data-space/operations/E4AFYQ44NL3UFXR4GJWEZSTA",
  "metadata": {
    "@type": "type.googleapis.com/google.earthengine.v1alpha.OperationMetadata",
    "state": "SUCCEEDED",
    "description": "Ingest image: \"projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017/S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631\"",
    "createTime": "2026-08-24T23:37:23.749341Z",
    "updateTime": "2026-08-24T23:55:48.139684Z",
    "startTime": "2026-08-24T23:37:32.413778Z",
    "endTime": "2026-08-24T23:55:48.139684Z",
    "type": "INGEST_IMAGE",
    "destinationUris": [
      "https://code.earthengine.google.com/?asset=projects/bop-nca-data-space/assets/S2_C1_L2A_2016_2017/S2A_MSIL2A_20160103T185122_N0500_R070_T11TNH_20231014T014631"
    ],
    "attempt": 1
  },
  "done": true,
  "response": {
    "@type": "type.googleapis.com/google.protobuf.Empty"
  }
}


## 11. Verify test asset band structure

In [14]:
test_info = ee.data.getAsset(
    test_asset_id
)

test_band_ids = [
    band["id"]
    for band in test_info["bands"]
]

print("Bands:", test_band_ids)
print("Band count:", len(test_band_ids))

if test_band_ids != EXPECTED_BANDS:
    raise RuntimeError(
        "Test asset band structure does not match expected 12-band layout."
    )

print("Test asset 12-band structure PASSED.")


Bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL', 'QA60']
Band count: 12
Test asset 12-band structure PASSED.


## 12. Test the cloud-probability join

Uses the existing MGRS tile + five-minute acquisition-time join.


In [15]:
def add_parsed_cloud_tile(image):
    image = ee.Image(image)

    parts = ee.String(
        image.get("system:index")
    ).split("_")

    last = ee.String(
        parts.get(
            parts.length().subtract(1)
        )
    )

    return image.set(
        "MGRS_TILE_PARSED",
        last.slice(1),
    )


img = ee.Image(test_asset_id)

test_date = ee.Date(
    img.get("system:time_start")
)

cloud_candidates = (
    ee.ImageCollection(
        "COPERNICUS/S2_CLOUD_PROBABILITY"
    )
    .filterDate(
        test_date.advance(-10, "minute"),
        test_date.advance(10, "minute"),
    )
    .filterBounds(
        img.geometry()
    )
    .map(
        add_parsed_cloud_tile
    )
)

print(
    "Cloud candidates:",
    cloud_candidates.size().getInfo()
)

print(
    "Candidate IDs:",
    cloud_candidates
    .aggregate_array("system:index")
    .getInfo()
)


join_filter = ee.Filter.And(
    ee.Filter.equals(
        leftField="MGRS_TILE",
        rightField="MGRS_TILE_PARSED",
    ),
    ee.Filter.maxDifference(
        difference=5 * 60 * 1000,
        leftField="system:time_start",
        rightField="system:time_start",
    ),
)


joined = ee.ImageCollection(
    ee.Join.saveFirst(
        "cloudprob"
    ).apply(
        primary=ee.ImageCollection([img]),
        secondary=cloud_candidates,
        condition=join_filter,
    )
)

joined_first = ee.Image(
    joined.first()
)

cloud_match_found = (
    joined_first.get("cloudprob").getInfo()
    is not None
)

print(
    "Historical tile:",
    img.get("MGRS_TILE").getInfo()
)

print(
    "Cloud match found:",
    cloud_match_found
)

if not cloud_match_found:
    raise RuntimeError(
        "Test cloud-probability join failed."
    )


Cloud candidates: 8
Candidate IDs: ['20160103T185122_20160103T185116_T11TNH', '20160103T185122_20160103T185116_T11TNJ', '20160103T185122_20160103T185116_T11TPH', '20160103T185122_20160103T185116_T11TPJ', '20160103T185246_20160119T162332_T11TNH', '20160103T185246_20160119T162332_T11TNJ', '20160103T185246_20160119T162332_T11TPH', '20160103T185246_20160119T162332_T11TPJ']
Historical tile: 11TNH
Cloud match found: True


## 13. Bulk re-ingestion

Enable only after the test asset and cloud-probability join pass.


In [ ]:
RUN_BULK_INGEST = True
SUBMIT_PAUSE_SECONDS = 2.0

bulk_results = []

if not RUN_BULK_INGEST:
    print("Bulk ingestion disabled.")
else:
    for _, row in tqdm(
        manifest_df.iterrows(),
        total=len(manifest_df),
        desc="Submitting",
    ):
        manifest = json.loads(
            Path(
                row.manifest_path
            ).read_text(
                encoding="utf-8"
            )
        )

        asset_id = manifest["name"]

        if asset_exists(asset_id):
            bulk_results.append({
                "product_id": row.product_id,
                "status": "skipped_existing",
                "operation": None,
                "error": None,
            })
            continue

        try:
            response = ee.data.startIngestion(
                None,
                manifest,
            )

            bulk_results.append({
                "product_id": row.product_id,
                "status": "submitted",
                "operation": response.get("name"),
                "error": None,
            })

        except Exception as exc:
            bulk_results.append({
                "product_id": row.product_id,
                "status": "submission_failed",
                "operation": None,
                "error": repr(exc),
            })

        time.sleep(
            SUBMIT_PAUSE_SECONDS
        )

    bulk_results_df = pd.DataFrame(
        bulk_results
    )

    display(
        bulk_results_df.status
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

    log_path = (
        MANIFEST_DIR
        / "bulk_ingestion_submissions_qa60.csv"
    )

    bulk_results_df.to_csv(
        log_path,
        index=False,
    )

    print(
        "Saved submission log:",
        log_path
    )


Submitting:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,submitted,561
1,skipped_existing,1


Saved submission log: A:\NCA_DATA\S2_2016-2017\ee_manifests_qa60\bulk_ingestion_submissions_qa60.csv


Submitting:   0%|          | 0/562 [00:00<?, ?it/s]

,status,count
0,skipped_existing,561
1,submitted,1


Saved submission log: A:\NCA_DATA\S2_2016-2017\ee_manifests_qa60\bulk_ingestion_submissions_qa60.csv


## 14. Monitor ingestion operations

In [19]:
ops = ee.data.listOperations(1000)

operation_rows = []

for op in ops:
    metadata = op.get("metadata", {})

    operation_rows.append({
        "name": op.get("name"),
        "done": op.get("done", False),
        "state": metadata.get("state"),
        "description": metadata.get("description"),
        "error": op.get("error"),
    })

operations_df = pd.DataFrame(
    operation_rows
)

if len(operations_df):
    display(
        operations_df.state
        .value_counts(dropna=False)
        .rename_axis("state")
        .reset_index(name="count")
    )

    display(
        operations_df.head(50)
    )


TypeError: 'int' object is not iterable

## 15. Final Earth Engine collection QA

In [ ]:
historical = ee.ImageCollection(
    EE_COLLECTION_ID
)

collection_count = (
    historical.size().getInfo()
)

print(
    "Collection count:",
    collection_count
)

print(
    "Expected:",
    EXPECTED_COUNT
)

first_bands = None

if collection_count > 0:
    first = ee.Image(
        historical.first()
    )

    first_bands = (
        first.bandNames().getInfo()
    )

    print(
        "First image bands:",
        first_bands
    )

    print(
        "First image date:",
        ee.Date(
            first.get("system:time_start")
        )
        .format("YYYY-MM-dd HH:mm:ss")
        .getInfo()
    )

    print(
        "First image tile:",
        first.get("MGRS_TILE").getInfo()
    )

if (
    collection_count == EXPECTED_COUNT
    and first_bands == EXPECTED_BANDS
):
    print(
        "Final collection structure PASSED."
    )
else:
    print(
        "Final collection is not yet complete."
    )


Collection count: 561
Expected: 562
First image bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL', 'QA60']
First image date: 2016-01-03 18:51:22
First image tile: 11TNH
Final collection is not yet complete.


Collection count: 561
Expected: 562
First image bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL', 'QA60']
First image date: 2016-01-03 18:51:22
First image tile: 11TNH
Final collection is not yet complete.


## 16. Full 2016–2017 cloud-probability join QA

In [47]:
def cloud_join_count_for_year(year):

    start = f"{year}-01-01"
    end   = f"{year + 1}-01-01"

    historical_year = (
        ee.ImageCollection(
            EE_COLLECTION_ID
        )
        .filterDate(
            start,
            end,
        )
    )

    clouds_year = (
        ee.ImageCollection(
            "COPERNICUS/S2_CLOUD_PROBABILITY"
        )
        .filterDate(
            start,
            end,
        )
        .map(
            add_parsed_cloud_tile
        )
    )

    join_filter = ee.Filter.And(

        ee.Filter.equals(
            leftField="MGRS_TILE",
            rightField="MGRS_TILE_PARSED",
        ),

        ee.Filter.maxDifference(
            difference=5 * 60 * 1000,
            leftField="system:time_start",
            rightField="system:time_start",
        ),
    )

    matched = ee.ImageCollection(
        ee.Join.saveFirst(
            "cloudprob"
        ).apply(
            primary=historical_year,
            secondary=clouds_year,
            condition=join_filter,
        )
    )

    total = historical_year.size().getInfo()
    matched_count = matched.size().getInfo()

    return {
        "year": year,
        "historical": total,
        "matched": matched_count,
        "unmatched": total - matched_count,
    }


results = []

for year in [2016, 2017]:

    print(
        "Checking",
        year,
        "..."
    )

    result = cloud_join_count_for_year(
        year
    )

    results.append(
        result
    )

    print(
        result
    )


qa = pd.DataFrame(
    results
)

display(
    qa
)

print(
    "\nTOTAL"
)

print(
    "Historical:",
    qa["historical"].sum()
)

print(
    "Matched:",
    qa["matched"].sum()
)

print(
    "Unmatched:",
    qa["unmatched"].sum()
)

Checking 2016 ...


KeyboardInterrupt: 

In [ ]:
def unmatched_cloudprob_for_year(year):

    start = f"{year}-01-01"
    end   = f"{year + 1}-01-01"

    historical_year = (
        ee.ImageCollection(
            EE_COLLECTION_ID
        )
        .filterDate(start, end)
    )

    clouds_year = (
        ee.ImageCollection(
            "COPERNICUS/S2_CLOUD_PROBABILITY"
        )
        .filterDate(start, end)
        .map(add_parsed_cloud_tile)
    )

    join_filter = ee.Filter.And(

        ee.Filter.equals(
            leftField="MGRS_TILE",
            rightField="MGRS_TILE_PARSED",
        ),

        ee.Filter.maxDifference(
            difference=5 * 60 * 1000,
            leftField="system:time_start",
            rightField="system:time_start",
        ),
    )

    unmatched = ee.ImageCollection(
        ee.Join.inverted().apply(
            primary=historical_year,
            secondary=clouds_year,
            condition=join_filter,
        )
    )

    return unmatched


for year in [2016, 2017]:

    unmatched = unmatched_cloudprob_for_year(
        year
    )

    print(
        f"\n{year} unmatched:",
        unmatched.size().getInfo()
    )

    ids = unmatched.aggregate_array(
        "PRODUCT_ID"
    ).getInfo()

    tiles = unmatched.aggregate_array(
        "MGRS_TILE"
    ).getInfo()

    times = unmatched.aggregate_array(
        "system:time_start"
    ).getInfo()

    for pid, tile, t in zip(
        ids,
        tiles,
        times,
    ):

        print(
            tile,
            ee.Date(t)
              .format("YYYY-MM-dd HH:mm:ss")
              .getInfo(),
            pid,
        )


2016 unmatched: 15
11TNJ 2016-01-23 18:51:12 S2A_MSIL2A_20160123T185112_N0500_R070_T11TNJ_20231014T092712
11TNH 2016-02-29 18:33:02 S2A_MSIL2A_20160229T183302_N0500_R027_T11TNH_20231015T223626
11TNJ 2016-02-29 18:33:02 S2A_MSIL2A_20160229T183302_N0500_R027_T11TNJ_20231015T223626
11TPH 2016-02-29 18:33:02 S2A_MSIL2A_20160229T183302_N0500_R027_T11TPH_20231015T223626
11TPH 2016-04-16 18:22:52 S2A_MSIL2A_20160416T182252_N0500_R127_T11TPH_20231020T120408
11TNH 2016-05-02 18:39:22 S2A_MSIL2A_20160502T183922_N0500_R070_T11TNH_20231003T203143
11TNJ 2016-05-02 18:39:22 S2A_MSIL2A_20160502T183922_N0500_R070_T11TNJ_20231003T203143
11TPH 2016-05-02 18:39:22 S2A_MSIL2A_20160502T183922_N0500_R070_T11TPH_20231003T203143
11TPH 2016-07-21 18:39:22 S2A_MSIL2A_20160721T183922_N0500_R070_T11TPH_20231022T155514
11TPH 2016-07-31 18:51:22 S2A_MSIL2A_20160731T185122_N0500_R070_T11TPH_20231006T112901
11TPH 2016-08-10 18:39:22 S2A_MSIL2A_20160810T183922_N0500_R070_T11TPH_20231022T055948
11TPH 2016-08-20 18:51: